In [4]:
import pandas as pd

In [5]:
"""
Hello World: USDA NASS QuickStats API
======================================
County-level crop statistics — yield, harvested area, irrigation.
Requires a free API key.
 
Registration (instant, free): https://quickstats.nass.usda.gov/api
 
Once registered, replace YOUR_NASS_API_KEY below with your key,
or set the environment variable NASS_API_KEY before running.
 
Useful stat categories:
  YIELD          — crop yield (e.g. BU / ACRE, CWT / ACRE)
  AREA HARVESTED — harvested acreage
  PRODUCTION     — total production volume
  AREA IRRIGATED — irrigated acreage (Census of Agriculture years only)
 
API docs: https://quickstats.nass.usda.gov/api
 
Dependencies:
    pip install requests
"""
 
import os
import requests
 
# Set your API key here or via the NASS_API_KEY environment variable
NASS_API_KEY = os.environ.get("NASS_API_KEY", "D010FDD0-1F8C-3CCA-8BFA-7345E97BD987")
 
 
def fetch_nass(
    commodity="RICE",
    state="ARKANSAS",
    year="2022",
    stat_cat="YIELD",
    agg_level="COUNTY",
    extra_params=None,
):
    """
    Fetch crop statistics from USDA NASS QuickStats.
 
    Args:
        commodity:    Crop name in uppercase (e.g. "RICE", "CORN", "SOYBEANS")
        state:        Full state name in uppercase (e.g. "ARKANSAS")
        year:         4-digit year as string
        stat_cat:     Statistic category (e.g. "YIELD", "AREA HARVESTED")
        agg_level:    Aggregation level: "COUNTY", "STATE", or "NATIONAL"
        extra_params: Optional dict of additional API parameters to merge in
 
    Returns:
        List of record dicts, or None if API key is not set.
    """
    if NASS_API_KEY == "YOUR_NASS_API_KEY":
        print("⚠  API key not set.")
        print("   Register at: https://quickstats.nass.usda.gov/api")
        print("   Then set NASS_API_KEY in this file or as an environment variable.")
        return None
 
    url = "https://quickstats.nass.usda.gov/api/api_GET/"
    params = {
        "key": NASS_API_KEY,
        "commodity_desc": commodity,
        "state_name": state,
        "year": year,
        "statisticcat_desc": stat_cat,
        "agg_level_desc": agg_level,
        "format": "JSON",
    }
    if extra_params:
        params.update(extra_params)
 
    print("── USDA NASS QuickStats ────────────────────────────────")
    print(f"Commodity: {commodity} | State: {state} | Year: {year}")
    print(f"Statistic: {stat_cat} | Level: {agg_level}")
    if extra_params:
        print(f"Extra:     {extra_params}")
    print()
 
    resp = requests.get(url, params=params, timeout=30)
 
    # NASS returns 400 when the query matches no data — inspect the body before raising
    if resp.status_code == 400:
        print(f"400 Bad Request — likely no data for this query combination.")
        try:
            print(f"API message: {resp.json()}")
        except Exception:
            print(f"Raw response: {resp.text[:300]}")
        return []
 
    resp.raise_for_status()
 
    records = resp.json().get("data", [])
    print(f"{len(records)} records returned.\n")
    return records
 
 
def print_summary(records, n=10):
    """Print a formatted table of the first n records."""
    if not records:
        print("No records to display.\n")
        return
    print(f"{'County':<25} {'Value':>12}  Unit")
    print("-" * 55)
    for rec in records[:n]:
        county = rec.get("county_name", "N/A")
        value  = rec.get("Value", "N/A")
        unit   = rec.get("unit_desc", "")
        print(f"{county:<25} {value:>12}  {unit}")
    if len(records) > n:
        print(f"  ... and {len(records) - n} more records")
    print()

In [6]:
records = fetch_nass(commodity="CORN", state="IOWA")
df = pd.DataFrame(records)

── USDA NASS QuickStats ────────────────────────────────
Commodity: CORN | State: IOWA | Year: 2022
Statistic: YIELD | Level: COUNTY

135 records returned.



In [8]:
df.columns

Index(['watershed_code', 'state_ansi', 'location_desc', 'asd_code', 'asd_desc',
       'country_name', 'year', 'end_code', 'region_desc', 'util_practice_desc',
       'agg_level_desc', 'commodity_desc', 'zip_5', 'county_ansi',
       'sector_desc', 'reference_period_desc', 'short_desc', 'group_desc',
       'week_ending', 'unit_desc', 'domaincat_desc', 'state_name', 'freq_desc',
       'country_code', 'state_fips_code', 'county_name', 'begin_code',
       'load_time', 'source_desc', 'county_code', 'watershed_desc',
       'prodn_practice_desc', 'statisticcat_desc', 'congr_district_code',
       'domain_desc', 'state_alpha', 'Value', 'CV (%)', 'class_desc'],
      dtype='str')